# Apache Iceberg Crypto Lakehouse Exploration

This notebook demonstrates how to query and explore the **Apache Iceberg** tables
managed via the REST Catalog and stored in MinIO (`s3://warehouse/`).

### Technology Stack:
- **Engine**: Apache Spark 3.5
- **Format**: Apache Iceberg
- **Catalog**: Iceberg REST Catalog (`http://iceberg-rest:8181`)
- **Storage**: MinIO (`http://minio:9000`)
- **Layers**: Bronze $\rightarrow$ Silver $\rightarrow$ Curated Gold

In [ ]:
from pyspark.sql import SparkSession

# SparkSession is pre-configured with Iceberg REST catalog in this environment
spark = (
    SparkSession.builder.appName("IcebergExploration")
    .getOrCreate()
)

print("Spark Version:", spark.version)
print("Current Catalog:", spark.catalog.currentCatalog())

## 1. Inspect Iceberg Catalogs & Namespaces

In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show()

## 2. Query Curated Gold Layer
Curated Gold is the authoritative reconciled dataset with priority resolution ($3 > 2 > 1$).

In [ ]:
gold_df = spark.table("lakehouse.gold.curated_crypto_bars")
gold_df.printSchema()
gold_df.show(10, truncate=False)

## 3. Query Daily OHLCV Analytical Views

In [ ]:
daily_view = spark.table("lakehouse.gold.v_daily_crypto_ohlcv")
daily_view.show(10, truncate=False)

## 4. Iceberg Metadata: Inspect Snapshots & History (Time Travel)

In [ ]:
# View table snapshots
spark.sql("SELECT snapshot_id, committed_at, operation, summary FROM lakehouse.gold.curated_crypto_bars.snapshots").show(truncate=False)

# View table history
spark.sql("SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor FROM lakehouse.gold.curated_crypto_bars.history").show(truncate=False)

## 5. Inspect Pipeline Audit Logs

In [ ]:
spark.table("lakehouse.gold.pipeline_audit").orderBy("created_at", ascending=False).show(truncate=False)